# 2019 서울 대학가 개강 여파 분석: **주차별 비교 최종본**

## 이 노트북에서 검증하는 질문
> **3월 개강 이후 대학 주변 역의 평일 승차량은, 개강 전 이용 특성이 유사한 비교역보다 더 크게 증가했는가?**

이번 최종본의 **주 분석 단위는 주(週)**입니다. 전체 과정을 한 파일에서 순서대로 실행합니다.

1. 2019년 이전에 개교한 서울 소재 일반대학·교육대학 캠퍼스 추출
2. 대학에서 **1km 이내**, 개강 전 평일 승차 평균 **1,000명 이상**인 역 후보
3. 모든 관련 대학·전문대학에서 **2km 이상** 떨어진 비교역 후보
4. 추가 매칭: **역간 거리 5km 이내**, 사전 승차 규모 **0.67~1.50배**, 6개 표준화 특성 차이 **RMS ≤ 1.0**, 가장 작은 RMS 선택
5. **역별 일일 승차량을 개강 전 평균=100으로 정규화**, 대학가·비교역 집단의 주차별 평균 이용지수 비교
6. **개강 전 주별 평균 vs 개강 후 주별 평균**, 각 집단 변화량과 **상대적 변화량 차이** 산출
7. **주별 차이의 HAC 회귀검정**, 그래프와 95% 신뢰구간
8. 보조검증: 일별 회귀 및 **월별 승차량 중앙값** 비교

> **범위 주의**: 이전 버전 코드의 `43 → 70/193 → 33/21`을 재현하려면 `ONLY_LINES_1_TO_9 = False`로 실행합니다. 기존 코드는 교통카드 CSV의 **다른 노선도 포함**했기 때문입니다. 1~9호선으로 엄격히 제한하려면 `True`로 바꿉니다. 두 범위를 같은 결과라고 기술하면 안 됩니다.

> **해석 주의**: 이 결과는 대학 개강과 이용량 변화의 **연관성**을 분석합니다. 지하철 승차자료는 실제 대학생을 구분하지 않으므로 인과적 영향으로 단정하지 않습니다.


## 1. 라이브러리 · 자동 파일 검색 · 분석 조건

Jupyter Notebook이 있는 폴더를 기준으로 파일을 검색합니다. 예를 들어 다음 구조를 지원합니다.

```text
실습/
├─ 이_노트북.ipynb
├─ 교육부_대학교 주소기반 좌표정보_20251126.xlsx
├─ 지하철노선위경도정보2.csv
└─ rawfiles/
   ├─ CARD_SUBWAY_MONTH_201901.csv
   └─ ...201906.csv
```

또한 파일명 끝에 `(1)`이 붙어 있어도 찾습니다. **이 코드는 `STATION_FILE`을 사용하므로 좌표 파일을 잘못 `rawfiles/`에서 찾는 오류가 없습니다.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from pathlib import Path
import platform

DATA_DIR = Path.cwd() / "data" / "opening"     # 대학 좌표 xlsx, 지하철노선위경도정보 csv
RAW_DIR = Path.cwd() / "data" / "professor"   # CARD_SUBWAY_MONTH_201901~201906.csv (수업 제공)
OUTPUT_DIR = Path.cwd() / "out" / "opening"
OUTPUT_DIR.parent.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# False: 이전 버전 코드와 동일한 전체 교통카드 CSV 노선 범위 (70 → 33개 재현)
# True : 노선명이 '1~9호선'인 데이터만 엄격히 사용
ONLY_LINES_1_TO_9 = False

START_DATE = pd.Timestamp("2019-01-01")
OPEN_DATE = pd.Timestamp("2019-03-04")
SEMESTER_END = OPEN_DATE + pd.Timedelta(weeks=16) - pd.Timedelta(days=1)

UNIVERSITY_RADIUS_M = 1000
CONTROL_RADIUS_M = 2000
MIN_PRE_BOARDINGS = 1000
MATCH_DISTANCE_M = 5000
SIZE_RATIO_MIN, SIZE_RATIO_MAX = 0.67, 1.50
MAX_MATCH_RMS = 1.0
MIN_DAYS_PER_WEEK = 3

# 존재하는 파일 중 첫 번째를 선택: root 우선, 파일명 숫자 (1)도 지원
def find_source(pattern, folders):
    matches = []
    for folder in folders:
        if folder.exists():
            matches += sorted(folder.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"파일을 찾지 못했습니다: {pattern} (검색 위치: {folders})")
    # 정확한 이름의 파일 우선, 중복 다운로드 표시 (1) 등은 다음 순위
    matches.sort(key=lambda p: ('(' in p.stem, len(p.stem)))
    if len(matches) > 1:
        print(f"[안내] '{pattern}' 후보 {len(matches)}개, 사용: {matches[0].name}")
    return matches[0]

UNIVERSITY_FILE = find_source("교육부_대학교 주소기반 좌표정보*.xlsx", [DATA_DIR, RAW_DIR])
STATION_FILE = find_source("지하철노선위경도정보*.csv", [DATA_DIR, RAW_DIR])
CARD_FILES = []
for month in range(1, 7):
    CARD_FILES.append(find_source(f"CARD_SUBWAY_MONTH_2019{month:02d}*.csv", [RAW_DIR, DATA_DIR]))

print("현재 작업 위치:", DATA_DIR)
print("대학교 좌표:", UNIVERSITY_FILE)
print("지하철 좌표:", STATION_FILE)
print("교통카드 파일:", [p.name for p in CARD_FILES])
print("노선 범위:", "1~9호선만" if ONLY_LINES_1_TO_9 else "전체 CSV 노선")
print("분석 기간:", START_DATE.date(), "~", SEMESTER_END.date())

# 윈도우 Jupyter에서 한국어 폰트 깨짐 방지
import matplotlib.font_manager as _fm
_names = {f.name for f in _fm.fontManager.ttflist}
plt.rcParams["font.family"] = next((f for f in ["Malgun Gothic", "AppleGothic", "NanumGothic"] if f in _names), next((f for f in sorted(_names) if "CJK" in f), "DejaVu Sans"))
plt.rcParams["axes.unicode_minus"] = False


## 2. 서울 소재 대학 캠퍼스 선정

교육부 좌표자료의 `학교상태`, `학교개교일`, `지역`, `학교구분`, `학제`를 활용합니다. `seoul_campuses`는 **대학가 역 1km 기준**에 사용하고, `exclusion_campuses`는 비교역을 일반대학·전문대학에서도 최소 2km 떨어뜨리기 위해 사용합니다.

2025년 대학 좌표를 2019년 분석에 활용하므로, 학교·캠퍼스 위치가 2019년과 달라졌을 가능성은 한계로 명시해야 합니다.

In [ ]:
university_raw = pd.read_excel(UNIVERSITY_FILE)
university_raw.columns = university_raw.columns.str.strip()

for column in ["학교명", "학교구분", "학제", "지역", "본분교", "학교상태"]:
    university_raw[column] = university_raw[column].astype("string").str.strip()
for column in ["위도", "경도"]:
    university_raw[column] = pd.to_numeric(university_raw[column], errors="coerce")
university_raw["개교일"] = pd.to_datetime(university_raw["학교개교일"], errors="coerce")

valid_base = (
    university_raw["학교상태"].eq("기존")
    & university_raw["개교일"].le(OPEN_DATE)
    & university_raw["위도"].between(-90, 90)
    & university_raw["경도"].between(-180, 180)
    & university_raw["위도"].ne(0)
    & university_raw["경도"].ne(0)
    & university_raw["학교명"].ne("KCUE대학교")
)
campus_base = university_raw.loc[valid_base].copy()

seoul_campuses = campus_base.loc[
    campus_base["지역"].eq("서울")
    & campus_base["학교구분"].eq("대학")
    & campus_base["학제"].isin(["대학교", "교육대학"])
].copy()

exclusion_campuses = campus_base.loc[
    campus_base["학교구분"].isin(["대학", "전문대학"])
    & ~campus_base["학제"].str.contains("사이버|방송통신", na=False)
].copy()

DEDUP_COLUMNS = ["학교명", "본분교", "위도", "경도"]
seoul_campuses = seoul_campuses.drop_duplicates(DEDUP_COLUMNS).reset_index(drop=True)
exclusion_campuses = exclusion_campuses.drop_duplicates(DEDUP_COLUMNS).reset_index(drop=True)
print("서울 소재 분석 캠퍼스 수:", len(seoul_campuses))
print("비교역 배제 반경 계산용 전국 캠퍼스 수:", len(exclusion_campuses))
display(seoul_campuses[["학교명", "본분교", "학제", "도로명 주소", "위도", "경도"]])


## 3. 2019년 1~6월 교통카드 승차량 데이터 전처리

- 동일한 이름의 환승역·여러 노선 관측치는 **날짜×역명 기준 합산**합니다. 뒤 단계에서 서로 멀리 떨어진 동명이역은 좌표 검사를 통해 제외합니다.
- 월~금만 사용하고 공휴일 및 5/1을 제외합니다.
- **개강 전 평균 승차량과 요일 패턴은 2019-01-01~2019-03-03 중 실제 관측된 평일만** 이용합니다. 비교역을 정할 때 개강 후 정보를 쓰지 않기 위해서입니다.
- 주 분석은 이전 버전 코드의 설정을 이어받아 2019-06-23까지만 사용합니다. 월별 중앙값 보조 분석은 6월 말까지 사용합니다.

In [ ]:
subway = pd.concat(
    [pd.read_csv(p, encoding="utf-8-sig") for p in CARD_FILES],
    ignore_index=True
)
subway.columns = subway.columns.str.strip()
print("원본에 있는 노선:", sorted(subway["노선명"].dropna().unique()))

if ONLY_LINES_1_TO_9:
    expected_lines = {f"{n}호선" for n in range(1, 10)}
    subway = subway.loc[subway["노선명"].isin(expected_lines)].copy()

subway["날짜"] = pd.to_datetime(subway["사용일자"].astype(str), format="%Y%m%d")
subway["역명"] = subway["역명"].astype("string").str.strip()
for column in ["승차총승객수", "하차총승객수"]:
    subway[column] = pd.to_numeric(subway[column], errors="raise")
    assert subway[column].notna().all() and subway[column].ge(0).all()

daily = subway.groupby(["날짜", "역명"], as_index=False)[
    ["승차총승객수", "하차총승객수"]
].sum()
assert not daily.duplicated(["날짜", "역명"]).any()

exclude_dates = pd.to_datetime([
    "2019-01-01", "2019-02-04", "2019-02-05", "2019-02-06",
    "2019-03-01", "2019-05-01", "2019-05-06", "2019-06-06"
])

university_daily = daily[["날짜", "역명", "승차총승객수"]].copy()
university_daily["요일번호"] = university_daily["날짜"].dt.dayofweek
university_daily = university_daily.loc[
    university_daily["날짜"].between(START_DATE, SEMESTER_END)
    & university_daily["요일번호"].lt(5)
    & ~university_daily["날짜"].isin(exclude_dates)
].copy()
university_daily["학기중"] = university_daily["날짜"].ge(OPEN_DATE).astype(int)

pre_data = university_daily.loc[university_daily["학기중"].eq(0)].copy()
pre_mean = pre_data.groupby("역명")["승차총승객수"].mean()
pre_weekday_pattern = (
    pre_data.pivot_table(index="역명", columns="요일번호", values="승차총승객수", aggfunc="mean")
    .reindex(columns=range(5))
    .div(pre_mean, axis=0)
)
print("원본 행 수:", len(subway), "/ 날짜별 역 집계 행 수:", len(daily))
print("개강 전 평일 관측 날짜 수:", pre_data["날짜"].nunique())
print("개강 후 평일 관측 날짜 수:", university_daily.loc[university_daily["학기중"].eq(1), "날짜"].nunique())
print("개강 전 규모 계산 가능한 역:", len(pre_mean))


## 4. Haversine 거리 계산 및 대학가·비교역 **후보** 추출

위·경도 차이를 구면 거리(m)로 바꿉니다. `distance_matrix`는 모든 역과 대학의 거리를 한 번에 계산합니다.

- **대학가 역 후보**: 서울 소재 선정 대학과 최단거리 ≤ 1,000m **그리고** 개강 전 평일 평균 ≥ 1,000명
- **비교역 후보**: 전국 관련 일반대학·전문대학과 최단거리 ≥ 2,000m **그리고** 개강 전 평일 평균 ≥ 1,000명
- 동일 역명이 다른 지역의 역을 가리켜 대표 좌표와 500m 이상 어긋나면 제외합니다.

이 단계에서는 아직 최종 1:1 비교역을 결정하지 않습니다.

In [ ]:
def distance_m(lat1, lon1, lat2, lon2):
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    a = (np.sin((lat2-lat1)/2)**2 +
         np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2)
    return 12742000 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def distance_matrix(left, right):
    return distance_m(
        left["위도"].to_numpy()[:, None], left["경도"].to_numpy()[:, None],
        right["위도"].to_numpy()[None, :], right["경도"].to_numpy()[None, :]
    )

station_raw = pd.read_csv(STATION_FILE, encoding="utf-8-sig").rename(columns={"역이름":"역명"})
station_raw.columns = station_raw.columns.str.strip()
station_raw["역명"] = station_raw["역명"].astype("string").str.strip()
for col in ["위도", "경도"]:
    station_raw[col] = pd.to_numeric(station_raw[col], errors="coerce")
station_raw = station_raw.loc[
    station_raw["역명"].isin(pre_mean.index)
    & station_raw["위도"].between(37, 38)
    & station_raw["경도"].between(126, 128)
].copy()

station_locations = station_raw.groupby("역명")[["위도", "경도"]].median().reset_index()
coordinate_check = station_raw.merge(station_locations, on="역명", suffixes=("", "_대표"))
coordinate_check["대표좌표와거리"] = distance_m(
    coordinate_check["위도"], coordinate_check["경도"],
    coordinate_check["위도_대표"], coordinate_check["경도_대표"]
)
ambiguous_names = coordinate_check.loc[coordinate_check["대표좌표와거리"].gt(500), "역명"].unique()
station_locations = station_locations.loc[~station_locations["역명"].isin(ambiguous_names)].copy()
print("동명이역 좌표 모호로 제외:", list(ambiguous_names))

univ_distance = distance_matrix(station_locations, seoul_campuses)
nearest_index = univ_distance.argmin(axis=1)
station_locations["최근접대학"] = seoul_campuses["학교명"].to_numpy()[nearest_index]
station_locations["대학거리_m"] = univ_distance.min(axis=1)
station_locations["학부캠퍼스거리_m"] = distance_matrix(station_locations, exclusion_campuses).min(axis=1)
station_locations["개강전평균"] = station_locations["역명"].map(pre_mean)
station_locations = station_locations.loc[station_locations["개강전평균"].ge(MIN_PRE_BOARDINGS)].copy()

treated_candidates = station_locations.loc[station_locations["대학거리_m"].le(UNIVERSITY_RADIUS_M)].copy()
control_candidates = station_locations.loc[station_locations["학부캠퍼스거리_m"].ge(CONTROL_RADIUS_M)].copy()

print("대학가 역 후보:", len(treated_candidates))
print("비교역 후보:", len(control_candidates))
display(treated_candidates[["역명","최근접대학","대학거리_m","개강전평균"]].round(1))


## 5. 추가 매칭 조건: 후보에서 실제 분석 대상으로 줄어드는 과정

여기서 처음으로 **각 대학가 역에 가장 유사한 비교역 1개**를 선택합니다. 순서는 다음과 같습니다.

1. **공간조건**: 대학가 역에서 직선거리 5km 이내 비교역만 남김
2. **규모조건**: 비교역의 개강 전 평일 평균 승차량이 대학가 역의 0.67~1.50배
3. **패턴조건**: 개강 전 `로그 승차량` 1개와 `월~금 상대 승차 패턴` 5개를 표준화해 차이의 RMS 점수 ≤ 1.0
4. 조건을 만족하는 역이 여러 개라면 **RMS 점수가 가장 작은 1개**를 선택

**70 → 33**은 매칭에 성공한 대학가 역 수이고, **193 → 21**은 전체 193개의 비교역 *후보* 중 최종 선택된 **서로 다른 비교역** 수입니다. **같은 비교역을 여러 대학가 역이 공유할 수 있습니다.** 선택되지 않은 나머지 비교역이 전부 기준 미달이라는 뜻은 아닙니다.

In [ ]:
features = pd.concat([
    np.log(pre_mean).rename("로그승차규모"),
    pre_weekday_pattern.rename(columns={i: f"요일패턴_{i}" for i in range(5)})
], axis=1)
features = features.loc[station_locations["역명"].tolist()]
assert not features.isna().any().any(), "개강 전 규모/요일 패턴에 결측이 있습니다."
standard_features = (features - features.mean()) / features.std(ddof=0).replace(0, 1)

pair_rows = []
match_audit_rows = []
for _, ts in treated_candidates.iterrows():
    station_name = ts["역명"]
    pool = control_candidates.copy()
    pool["역간거리_m"] = distance_m(ts["위도"],ts["경도"],pool["위도"],pool["경도"])
    near = pool.loc[pool["역간거리_m"].le(MATCH_DISTANCE_M)].copy()
    sized = near.loc[near["개강전평균"].between(
        ts["개강전평균"]*SIZE_RATIO_MIN, ts["개강전평균"]*SIZE_RATIO_MAX
    )].copy()

    valid_count = 0
    if len(sized):
        differences = standard_features.loc[sized["역명"].tolist()] - standard_features.loc[station_name]
        scores = np.sqrt((differences**2).mean(axis=1))
        valid_count = int((scores <= MAX_MATCH_RMS).sum())
    
    if not len(near):
        reason = "5km 이내 비교역 없음"
    elif not len(sized):
        reason = "승차 규모 0.67~1.5배 불충족"
    elif not valid_count:
        reason = "표준화 RMS > 1.0"
    else:
        reason = "매칭 성공"
        best_name = scores.idxmin()
        best = sized.loc[sized["역명"].eq(best_name)].iloc[0]
        pair_rows.append({
            "대학역": station_name,
            "최근접대학": ts["최근접대학"],
            "비교역": best_name,
            "대학역_개강전평균": ts["개강전평균"],
            "비교역_개강전평균": best["개강전평균"],
            "역간거리_m": best["역간거리_m"],
            "매칭차이": scores.loc[best_name]
        })
    match_audit_rows.append({
        "대학역": station_name,
        "5km 이내 후보 수": len(near),
        "승차 규모 조건 후보 수": len(sized),
        "RMS 기준 통과 후보 수": valid_count,
        "결과": reason
    })

pairs = pd.DataFrame(pair_rows)
match_audit = pd.DataFrame(match_audit_rows)
if pairs.empty:
    raise RuntimeError("비교역 매칭 성공 사례가 없습니다. 노선 범위와 조건을 점검하세요.")

print("대학가 역 후보:", len(treated_candidates), "→ 최종 대학가 역:", len(pairs))
print("비교역 후보:", len(control_candidates), "→ 고유 선택 비교역:", pairs["비교역"].nunique())
print("전체 대응쌍 수:", len(pairs), " / 재사용 비교역 수:", int(pairs["비교역"].value_counts().gt(1).sum()))
print("\n[매칭 결과별 개수]")
print(match_audit["결과"].value_counts().to_string())
assert len(treated_candidates) == len(pairs) + len(match_audit.loc[match_audit["결과"].ne("매칭 성공")])
display(pairs.round(2))
display(match_audit.sort_values(["결과", "대학역"]))
pairs.to_csv(OUTPUT_DIR / "01_대학가역_비교역_매칭표.csv", index=False, encoding="utf-8-sig")
match_audit.to_csv(OUTPUT_DIR / "02_매칭실패_단계별사유.csv", index=False, encoding="utf-8-sig")


## 6. 주차별 이용지수 만들기: **이 단계가 이번 분석의 핵심**

원래 승차량이 8만 명인 역과 1만 명인 역을 바로 평균하면 큰 역이 전체 결과를 좌우할 수 있습니다. 따라서 먼저 **각 역의 개강 전 평일 일평균 승차량을 100**으로 정규화합니다.

**역별 이용지수 = 100 × (해당 날짜 승차량 ÷ 해당 역의 개강 전 평일 평균 승차량)**

- **매일**: 대응쌍 33개 기준으로 대학가 역 이용지수를 평균하고, 대응 비교역 이용지수를 평균합니다. (같은 비교역이 여러 번 매칭됐다면 해당 쌍만큼 반영)
- **매주**: 해당 주의 *관측된 평일* 일별 집단 평균을 평균합니다.
- 관측일수 3일 미만의 불완전한 주는 제외합니다. `weekly_compare`를 이번 노트북에서 **직접 새로 생성**하므로 이전에 발생한 `KeyError: '대학인근'`과 같은 이름 충돌을 방지합니다.

> 이 주별 분석은 **주별 평균 이용지수**입니다. 승객 수의 중앙값은 아래 보조 분석에서 별도로 계산합니다.

In [ ]:
boarding_wide = university_daily.pivot(index="날짜", columns="역명", values="승차총승객수").sort_index()
needed_stations = sorted(set(pairs["대학역"]) | set(pairs["비교역"]))
if boarding_wide[needed_stations].isna().any().any():
    raise ValueError("분석 대상 역의 날짜별 승차량 누락: 필요한 파일이 모두 있는지 확인하세요.")

use_index = boarding_wide[needed_stations].div(pre_mean.loc[needed_stations], axis=1) * 100

group_daily = pd.DataFrame({
    "날짜": use_index.index,
    "대학인근": use_index[pairs["대학역"].tolist()].mean(axis=1).to_numpy(),
    "비교지역": use_index[pairs["비교역"].tolist()].mean(axis=1).to_numpy()
})
group_daily["그룹차이"] = group_daily["대학인근"] - group_daily["비교지역"]
group_daily["학기중"] = group_daily["날짜"].ge(OPEN_DATE).astype(int)
group_daily["요일번호"] = group_daily["날짜"].dt.dayofweek
group_daily["주시작일"] = (
    group_daily["날짜"] - pd.to_timedelta(group_daily["요일번호"], unit="D")
)

weekly_compare = (
    group_daily.groupby("주시작일", as_index=False)
    .agg(
        대학인근=("대학인근", "mean"),
        비교지역=("비교지역", "mean"),
        관측일수=("날짜", "nunique")
    )
)
weekly_compare = weekly_compare.loc[weekly_compare["관측일수"].ge(MIN_DAYS_PER_WEEK)].copy()
weekly_compare["개강후"] = weekly_compare["주시작일"].ge(OPEN_DATE).astype(int)
weekly_compare["기간"] = weekly_compare["개강후"].map({0: "개강 전", 1: "개강 후"})
weekly_compare["주별차이"] = weekly_compare["대학인근"] - weekly_compare["비교지역"]

# 아래 출력이 정상적이라면 앞선 KeyError는 발생하지 않습니다.
print("주별 자료 열:", weekly_compare.columns.tolist())
print("개강 전 주 수:", int(weekly_compare["개강후"].eq(0).sum()))
print("개강 후 주 수:", int(weekly_compare["개강후"].eq(1).sum()))
display(weekly_compare.round(2))
weekly_compare.to_csv(OUTPUT_DIR / "03_주별_대학가역_비교역_이용지수.csv", index=False, encoding="utf-8-sig")


## 7. **각 주를 먼저 계산한 다음**, 개강 전후 주별 평균 비교

계산 순서는 다음과 같습니다.

1. 개강 전의 여러 주에서 **대학 인근 역 주별 이용지수의 평균**과 **비교역 주별 이용지수의 평균**을 구함
2. 개강 후 여러 주에서도 각각 같은 방식으로 계산
3. 대학 인근 역 변화량: `개강 후 평균 − 개강 전 평균`
4. 비교역 변화량: `개강 후 평균 − 개강 전 평균`
5. 상대적 추가 변화: **대학 인근 역 변화량 − 비교역 변화량**

\[
\Delta=(\overline{U}_{후}-\overline{U}_{전})-(\overline{C}_{후}-\overline{C}_{전})
\]

이 값은 **이용지수점**(개강 전 역별 평균 100 기준)이지 승객 수나 엄밀한 퍼센트포인트 변화가 아닙니다. 한 주의 관측 일수가 3일인지 5일인지와 관계없이 각 주에 같은 가중치를 줍니다.

In [ ]:
summary_week = (
    weekly_compare.groupby("기간")[["대학인근", "비교지역", "주별차이"]]
    .mean()
    .reindex(["개강 전", "개강 후"])
)
assert summary_week.notna().all().all()

univ_change = summary_week.loc["개강 후", "대학인근"] - summary_week.loc["개강 전", "대학인근"]
control_change = summary_week.loc["개강 후", "비교지역"] - summary_week.loc["개강 전", "비교지역"]
relative_effect = univ_change - control_change

print("[개강 전후 주별 이용지수의 평균]")
display(summary_week.round(3))
print(f"대학가 역 변화: {univ_change:+.3f} 이용지수점")
print(f"비교역 변화: {control_change:+.3f} 이용지수점")
print(f"상대적 추가 변화: {relative_effect:+.3f} 이용지수점")

# 집단차이의 전후 평균을 뺀 계산과 동일한지 검산
assert np.isclose(relative_effect,
                  summary_week.loc["개강 후", "주별차이"]-summary_week.loc["개강 전", "주별차이"])
summary_week.to_csv(OUTPUT_DIR / "04_개강전후_주별평균_요약.csv", encoding="utf-8-sig")


## 8. **주별 자료 자체를 이용하는 통계검정** (주 분석)

주별 두 집단 차이를 종속변수로 하는 회귀모형을 사용합니다.

**주별 집단차이 = 상수 + 개강후 계수 × 개강후 여부 + 오차** (주별 집단차이 = 대학가 역 이용지수 − 비교역 이용지수)

- **H₀**: 개강 전후 평균적인 대학가–비교역 차이의 변화는 0이다 (β₁=0).
- **H₁**: 그 변화는 0이 아니다 (β₁≠0), 양측 검정.
- `HAC(maxlags=2)`는 인접 주차의 오차 상관을 부분적으로 고려하는 표준오차입니다.
- **β₁이 위에서 계산한 `relative_effect`와 같아야 정상**입니다. 값은 같더라도 검정에는 주차 수와 변동성이 반영됩니다.

다만 관측된 주가 많지 않고 다른 계절 요인이 남으므로, p-value가 작더라도 대학생 유입의 순수 인과효과로 단정하지 않습니다.

In [ ]:
weekly_fit = smf.ols(
    "주별차이 ~ 개강후", data=weekly_compare
).fit(cov_type="HAC", cov_kwds={"maxlags": 2})

weekly_effect = float(weekly_fit.params["개강후"])
weekly_p = float(weekly_fit.pvalues["개강후"])
weekly_lower, weekly_upper = weekly_fit.conf_int().loc["개강후"].tolist()
assert np.isclose(weekly_effect, relative_effect)

weekly_result = pd.DataFrame([{
    "대학가 역 수": pairs["대학역"].nunique(),
    "고유 비교역 수": pairs["비교역"].nunique(),
    "개강 전 주 수": int(weekly_compare["개강후"].eq(0).sum()),
    "개강 후 주 수": int(weekly_compare["개강후"].eq(1).sum()),
    "대학가 역 변화(지수점)": univ_change,
    "비교역 변화(지수점)": control_change,
    "상대적 증가(지수점)": weekly_effect,
    "95% CI 하한": weekly_lower,
    "95% CI 상한": weekly_upper,
    "양측 p-value": weekly_p
}])
print("[주별 차이에 대한 HAC 통계검정]")
display(weekly_result.round(4))
print(f"상대적 변화 {weekly_effect:+.3f} 지수점 / 95% CI [{weekly_lower:+.3f}, {weekly_upper:+.3f}] / p={weekly_p:.5f}")
weekly_result.to_csv(OUTPUT_DIR / "05_주별_HAC_통계검정.csv", index=False, encoding="utf-8-sig")


## 9. 주별 비교 그래프 · 개강 후 변화량 95% 신뢰구간

왼쪽은 **모든 주차**에 대해 대학 인근 역·매칭 비교역의 평균 이용지수를 연결한 그래프이며, 오른쪽 점과 막대는 **주별 회귀분석의 추정값과 95% 신뢰구간**입니다.

이전 노트북의 오른쪽 그래프는 **일별 회귀검정**이었지만, 이번 오른쪽 그래프는 **주별 평균 분석 자체와 일치하는 검정**의 결과입니다.

In [ ]:
# =====================================================
# 한글 글꼴 설정 (Windows)
# =====================================================

import matplotlib.font_manager as _fm
_names = {f.name for f in _fm.fontManager.ttflist}
plt.rcParams["font.family"] = next((f for f in ["Malgun Gothic", "AppleGothic", "NanumGothic"] if f in _names), next((f for f in sorted(_names) if "CJK" in f), "DejaVu Sans"))
plt.rcParams["axes.unicode_minus"] = False


# =====================================================
# 1. 주별 이용지수 변화 + 통계검정 결과
# =====================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 5),
    gridspec_kw={"width_ratios": [2, 1]}
)

# 왼쪽: 대학 인근 역
axes[0].plot(
    weekly_compare["주시작일"],
    weekly_compare["대학인근"],
    marker="o",
    ms=4,
    label="대학 인근 역",
    color="steelblue"
)

# 왼쪽: 매칭된 비교역
axes[0].plot(
    weekly_compare["주시작일"],
    weekly_compare["비교지역"],
    marker="o",
    ms=4,
    label="매칭된 비교역",
    color="darkorange"
)

# 개강 기준일
axes[0].axvline(
    OPEN_DATE,
    color="black",
    ls="--",
    lw=1.2,
    label="개강 기준일 (3월 4일)"
)

# 기준 이용지수 100
axes[0].axhline(
    100,
    color="gray",
    ls=":",
    alpha=0.7
)

axes[0].set_title(
    "개강 전후 대학 인근 역과 비교역의 주별 평일 이용지수"
)

axes[0].set_xlabel("주차 시작일")
axes[0].set_ylabel("이용지수 (개강 전 평균 = 100)")

axes[0].legend()
axes[0].grid(alpha=0.2)
axes[0].tick_params(axis="x", rotation=35)


# =====================================================
# 오른쪽: 상대적 개강 효과 및 95% 신뢰구간
# =====================================================

axes[1].errorbar(
    weekly_effect,
    0,
    xerr=np.array([
        [weekly_effect - weekly_lower],
        [weekly_upper - weekly_effect]
    ]),
    fmt="o",
    ms=8,
    capsize=6,
    color="steelblue"
)

# 상대적 변화가 0인 기준선
axes[1].axvline(
    0,
    color="black",
    ls="--"
)

axes[1].set_yticks([])
axes[1].set_ylim(-0.5, 0.5)

# p-value 표시
ptext = (
    "p < 0.001"
    if weekly_p < 0.001
    else f"p = {weekly_p:.4f}"
)

axes[1].set_title(
    f"개강 후 상대적 추가 증가: {weekly_effect:+.2f} 이용지수점\n"
    f"통계검정 결과: {ptext}"
)

axes[1].set_xlabel(
    "개강 전후 상대적 변화량 (95% 신뢰구간)"
)

axes[1].grid(axis="x", alpha=0.2)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "06_주별이용지수_개강전후_검정.png",
    dpi=170,
    bbox_inches="tight"
)

plt.show()


# =====================================================
# 2. 개강 전후 집단별 주평균 이용지수 막대그래프
# =====================================================

fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(2)
width = 0.35

# 대학 인근 역 막대
ax.bar(
    x - width / 2,
    summary_week["대학인근"],
    width,
    label="대학 인근 역",
    color="steelblue"
)

# 비교역 막대
ax.bar(
    x + width / 2,
    summary_week["비교지역"],
    width,
    label="매칭된 비교역",
    color="darkorange"
)

# 가로축: 개강 전 / 개강 후
ax.set_xticks(
    x,
    summary_week.index.tolist()
)

ax.set_ylabel("주별 평균 이용지수 (개강 전 평균 = 100)")
ax.set_xlabel("분석 기간")
ax.set_title("개강 전후 대학 인근 역과 비교역의 주별 평균 이용지수")

ax.grid(axis="y", alpha=0.2)
ax.legend()

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "07_개강전후_주별평균_집단비교.png",
    dpi=170,
    bbox_inches="tight"
)

plt.show()

## 10. 일별 회귀검정: **보조 결과**

이전 버전 코드의 `그룹차이 ~ 학기중 + C(요일번호)`를 사용하되, `주` 단위로 표준오차를 군집화합니다. 일별 데이터를 사용하므로 주별 분석과 계수·p-value가 다를 수 있습니다. **최종 보고서의 주 지표는 위 8번(주별 HAC)으로 통일**하고, 이 결과는 분석 단위가 달라도 경향이 유지되는지 확인하는 용도로 쓰세요.

In [ ]:
daily_regression = group_daily.copy()
daily_regression["주ID"] = daily_regression["주시작일"].dt.strftime("%Y-%m-%d")
daily_fit = smf.ols(
    "그룹차이 ~ 학기중 + C(요일번호)", data=daily_regression
).fit(cov_type="cluster",
      cov_kwds={"groups": daily_regression["주ID"].to_numpy(),
                "use_correction": True, "df_correction": True},
      use_t=True)
daily_effect=float(daily_fit.params["학기중"])
daily_lower,daily_upper=daily_fit.conf_int().loc["학기중"].tolist()
print(f"[보조: 일별 요일통제 회귀] 변화={daily_effect:+.3f} 지수점, 95% CI=[{daily_lower:+.3f}, {daily_upper:+.3f}], p={daily_fit.pvalues['학기중']:.5g}")


## 11. 월별 승차량 **중앙값** 비교: 보조 분석

이 분석은 주별 분석을 대체하지 않습니다. 홍대입구처럼 특정 날짜 방문객이 급증하는 역의 영향을 완화하기 위해 **역마다 월별 평일 일일 승차량 중앙값**을 구하고, 매칭쌍 기준으로 평균합니다.

- 1~6월 각각 대학가·비교역의 **중앙값 평균 그래프**
- 1~2월 월별 중앙값 평균을 개강 전, 3~6월을 개강 후로 묶어 **집단 증가율**을 계산

**주의**: 중앙값은 상시적으로 방문하는 외부인을 제거하지는 못합니다. 또 월별 중앙값 증가율과 주별 이용지수 효과는 서로 다른 통계량이므로 수치를 일치시킬 필요가 없습니다.

In [ ]:

# =====================================================
# 1. 월별 평일 승차량 중앙값 계산
# =====================================================

# 한글 글꼴 설정
import matplotlib.font_manager as _fm
_names = {f.name for f in _fm.fontManager.ttflist}
plt.rcParams["font.family"] = next((f for f in ["Malgun Gothic", "AppleGothic", "NanumGothic"] if f in _names), next((f for f in sorted(_names) if "CJK" in f), "DejaVu Sans"))
plt.rcParams["axes.unicode_minus"] = False

# 날짜, 역명, 승차량 데이터 추출
month_work = daily[
    ["날짜", "역명", "승차총승객수"]
].copy()

# 요일 계산 (월요일=0, 일요일=6)
month_work["요일번호"] = month_work["날짜"].dt.dayofweek

# 평일만 선택, 공휴일 제외, 1~6월 데이터 사용
month_work = month_work.loc[
    month_work["요일번호"].lt(5)
    & ~month_work["날짜"].isin(exclude_dates)
    & month_work["날짜"].dt.month.between(1, 6)
].copy()

# 월 정보 추가
month_work["월"] = month_work["날짜"].dt.month

# 역별·월별 평일 일일 승차량 중앙값 계산
monthly_median = (
    month_work.groupby(["역명", "월"])["승차총승객수"]
    .median()
    .unstack("월")
    .reindex(columns=range(1, 7))
)

# 누락 데이터 확인
if monthly_median.reindex(needed_stations).isna().any().any():
    raise ValueError("월별 중앙값에 누락된 역·월이 있습니다.")


# =====================================================
# 2. 대학가 역과 비교역의 월별 중앙값 평균
# =====================================================

monthly_rows = []

for month in range(1, 7):
    monthly_rows.append({
        "월": month,

        "대학가역_중앙값평균": monthly_median.loc[
            pairs["대학역"].tolist(), month
        ].mean(),

        "비교역_중앙값평균": monthly_median.loc[
            pairs["비교역"].tolist(), month
        ].mean()
    })

monthly_summary = pd.DataFrame(monthly_rows)

# 대학가 역과 비교역의 월별 승차량 차이
monthly_summary["두집단차이_명"] = (
    monthly_summary["대학가역_중앙값평균"]
    - monthly_summary["비교역_중앙값평균"]
)

print("① 월별 평일 승차량 중앙값 평균 비교")
display(monthly_summary.round(2))

# 분석 결과 CSV 저장
monthly_summary.to_csv(
    OUTPUT_DIR / "08_월별중앙값평균.csv",
    index=False,
    encoding="utf-8-sig"
)


# =====================================================
# 3. 월별 중앙값 평균 그래프 (한국어)
# =====================================================

fig, ax = plt.subplots(figsize=(10, 5))

# 대학가 역
ax.plot(
    monthly_summary["월"],
    monthly_summary["대학가역_중앙값평균"],
    marker="o",
    linewidth=2,
    label="대학 인근 역",
    color="steelblue"
)

# 매칭된 비교역
ax.plot(
    monthly_summary["월"],
    monthly_summary["비교역_중앙값평균"],
    marker="o",
    linewidth=2,
    label="매칭된 비교역",
    color="darkorange"
)

# 개강 시점 표시
ax.axvline(
    2.5,
    linestyle="--",
    color="gray",
    label="3월 개강"
)

# 가로축: 1월~6월
ax.set_xticks(
    range(1, 7),
    [f"{month}월" for month in range(1, 7)]
)

ax.set_xlabel("2019년 월")
ax.set_ylabel("평일 일일 승차량 중앙값의 평균 (명)")

ax.set_title(
    "2019년 1~6월 대학 인근 역과 비교역의 "
    "월별 승차량 중앙값 비교"
)

# 그래프의 각 데이터 값 표시
for column, color, offset in [
    ("대학가역_중앙값평균", "steelblue", 8),
    ("비교역_중앙값평균", "darkorange", -16)
]:
    for _, row in monthly_summary.iterrows():
        ax.annotate(
            f"{row[column]:,.0f}",
            (row["월"], row[column]),
            textcoords="offset points",
            xytext=(0, offset),
            ha="center",
            fontsize=8,
            color=color
        )

ax.grid(alpha=0.2)
ax.legend()

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "09_월별중앙값_보조그래프.png",
    dpi=170,
    bbox_inches="tight"
)

plt.show()


# =====================================================
# 4. 개강 전후 기간별 중앙값 평균 계산
# =====================================================

# 개강 전: 1~2월 중앙값 평균
monthly_pre = monthly_median[[1, 2]].mean(axis=1)

# 개강 후: 3~6월 중앙값 평균
monthly_post = monthly_median[[3, 4, 5, 6]].mean(axis=1)

# 대학가 역 - 비교역 매칭표
med_pairs = pairs[
    ["대학역", "비교역"]
].copy()

for group, column in [
    ("대학가", "대학역"),
    ("비교", "비교역")
]:

    # 개강 전 중앙값 평균
    med_pairs[f"{group}_전"] = (
        med_pairs[column].map(monthly_pre)
    )

    # 개강 후 중앙값 평균
    med_pairs[f"{group}_후"] = (
        med_pairs[column].map(monthly_post)
    )

    # 개강 전후 증가율
    med_pairs[f"{group}_증가율_pct"] = (
        (
            med_pairs[f"{group}_후"]
            / med_pairs[f"{group}_전"]
        ) - 1
    ) * 100

# 매칭쌍별 증가율 차이
med_pairs["상대증가율차_pctp"] = (
    med_pairs["대학가_증가율_pct"]
    - med_pairs["비교_증가율_pct"]
)


# =====================================================
# 5. 개강 전후 승차량 증가율 결과 출력
# =====================================================

print("\n② 개강 전후 월별 중앙값 평균 변화")

for group in ["대학가", "비교"]:

    before = med_pairs[f"{group}_전"].mean()
    after = med_pairs[f"{group}_후"].mean()

    growth = 100 * (after / before - 1)

    print(
        f"{group} 역: "
        f"개강 전 {before:,.1f}명 → "
        f"개강 후 {after:,.1f}명 "
        f"(증가율: {growth:+.2f}%)"
    )

print(
    "\n③ 매칭쌍별 상대적 증가율 차이 평균:",
    f"{med_pairs['상대증가율차_pctp'].mean():+.2f}%p"
)

print("\n④ 대학가 역별 중앙값 증가율 비교")
display(med_pairs.round(2))

# CSV 저장
med_pairs.to_csv(
    OUTPUT_DIR / "10_월중앙값_역별_증가율.csv",
    index=False,
    encoding="utf-8-sig"
)


## 12. 결과를 어떻게 보고서에 쓰면 될까?

**분석 대상 설명**

2019년 3월 개강 전후 서울 소재 일반대학·교육대학 인근 역과 유사 비교역의 평일 승차량 변화를 비교하였다. 대학에서 1km 이내이며 개강 전 일평균 승차량이 1,000명 이상인 역을 대학가 역 후보로, 일반대학·전문대학에서 최소 2km 이상 떨어진 역을 비교역 후보로 정하였다. 두 역 간 거리 5km, 사전 승차 규모 0.67~1.5배, 표준화된 사전 승차 규모·요일패턴 차이 RMS 1.0 이하의 기준을 적용해 비교역을 선정하였다.

**주별 분석 설명**

각 역의 개강 전 평일 일평균 승차량을 100으로 정규화한 다음, 대응쌍의 날짜별 평균 이용지수를 계산하였다. 이를 관측 평일이 3일 이상인 주별로 평균하여 대학가 역과 비교역의 주별 변화를 도식화하였다. 개강 전·후의 주별 집단 평균을 각각 구하고 두 집단의 변화량 차이를 산출했으며, 주별 차이에 HAC 표준오차를 적용한 회귀분석으로 유의성을 검정하였다.

**주의할 제한**

- **관측 수는 서로 독립인 대학생 수가 아닙니다.** 같은 비교역이 여러 대학가 역에 매칭되며 역별 관측값도 시간에 따라 상관될 수 있습니다.
- **1~2월 vs 3~6월** 차이에는 계절 요인, 학교별 개강·종강 시점 차이, 상업·관광 수요 등이 섞일 수 있습니다.
- 일별·주별·월중앙값 분석의 **추정값과 p-value는 같은 통계량이 아닙니다.** 최종 보고서의 핵심 표·p-value는 **8번 주별 HAC 결과**를 우선 사용하세요.
- 사전 패턴이 비슷하다는 조건만으로 **인과적 평행추세 가정**이 완전히 확인된 것은 아닙니다.

**저장 결과**: `개강_분석결과/` 폴더에 매칭표, 매칭 실패 이유, 주별 값, 개강 전후 요약, 주별 통계검정, 그래프, 월별 중앙값 표가 생성됩니다. 다음 단계의 *상업지역 영향 분석*에서는 매칭표와 역별 결과를 그대로 활용하면 됩니다.


# 추가 분석: 기존 EDA 역 유형에 따른 대학가 개강 효과 차이

**연구 질문**: 기존 EDA에서 정한 업무·상업·관광·주거 유형의 역 가운데, 대학가 역으로도 선정된 역에서 개강 이후 상대적 승차량 변화가 유형마다 다른가?

- 이 분석은 **기존 최종본의 `pairs`, `use_index`, `OPEN_DATE`, `MIN_DAYS_PER_WEEK`, `OUTPUT_DIR`를 재사용**합니다.
- 36개 역 유형표를 기존의 33개 최종 대학가 역에 연결합니다. **사전에 분류되지 않은 역을 억지로 분류하지 않습니다.**
- 중복 집계하지 않도록 대학가 역마다 대응하는 비교역의 주별 이용지수 차이를 사용합니다.
- 유형별 표본이 2개 역씩인 점과 주거 유형에서 **수락산 비교역을 공유**하는 점을 고려하여 **탐색적 기술통계 중심**으로 해석합니다. 확인적인 p-value를 제시하지 않습니다.


In [ ]:
# ===============================================================
# 추가 1. 기존 EDA의 36개 역 유형과 대학가 역 33개 연결
# ===============================================================
import matplotlib.font_manager as _fm
_names = {f.name for f in _fm.fontManager.ttflist}
plt.rcParams["font.family"] = next((f for f in ["Malgun Gothic", "AppleGothic", "NanumGothic"] if f in _names), next((f for f in sorted(_names) if "CJK" in f), "DejaVu Sans"))

station_groups = {
    "업무": ["강남", "역삼", "선릉", "삼성", "여의도", "종각", "시청", "을지로입구", "광화문", "구로디지털단지"],
    "상업": ["홍대입구", "신촌", "이태원", "명동", "건대입구", "합정", "압구정", "동대문역사문화공원"],
    "관광": ["경복궁", "안국", "대공원", "어린이대공원", "뚝섬유원지", "여의나루", "올림픽공원", "경마공원"],
    "주거": ["상계", "중계", "하계", "마들", "수락산", "방화", "고덕", "명일", "마천", "거여"]
}
assert sum(map(len, station_groups.values())) == 36
long_labels = [(name, category) for category, names in station_groups.items() for name in names]
type_map = pd.DataFrame(long_labels, columns=["역명", "역유형"])
assert not type_map["역명"].duplicated().any(), "역 유형 목록에 중복 역명이 있습니다."

# 원본 pairs는 변경하지 않고, 유형만 추가한 사본을 생성
pairs_type = pairs.copy()
pairs_type["대학역"] = pairs_type["대학역"].astype(str).str.strip()
pairs_type["비교역"] = pairs_type["비교역"].astype(str).str.strip()
pairs_type = pairs_type.merge(type_map, left_on="대학역", right_on="역명", how="left", validate="many_to_one").drop(columns="역명")
pairs_type["역유형"] = pairs_type["역유형"].fillna("미분류")

print("고유 대학가 역:", pairs_type["대학역"].nunique())
print("사전 분류에 포함된 대학가 역:", pairs_type.loc[pairs_type["역유형"] != "미분류", "대학역"].nunique())
print("유형별 역 수:")
counts = pairs_type["역유형"].value_counts().reindex(["업무", "상업", "관광", "주거", "미분류"], fill_value=0)
display(counts.rename("역 수").to_frame())
display(pairs_type.loc[pairs_type["역유형"] != "미분류", ["역유형", "대학역", "비교역", "최근접대학"]].sort_values(["역유형", "대학역"]))

# 전체 표본 대비 분류 범위 그래프
fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#7B8D9D", "#F18D42", "#31A68A", "#6088CB", "#B3B3B3"]
ax.bar(counts.index, counts.values, color=colors)
for x, y in enumerate(counts.values):
    ax.text(x, y + 0.4, str(y), ha="center", va="bottom", fontsize=11)
ax.set_ylim(0, max(counts.values) * 1.18)
ax.set_title("대학가 역 33개 중 기존 역 유형표에 포함된 역 수")
ax.set_ylabel("대학가 역 수")
ax.grid(axis="y", alpha=.2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "13_기존유형과_대학가역_교집합.png", dpi=170)
plt.show()

# 업무 유형 0, 다른 유형 표본이 작은 경우 검정 결과 일반화에 주의
classified_pairs = pairs_type.loc[pairs_type["역유형"].ne("미분류")].copy()
if classified_pairs.empty:
    raise ValueError("기존 역 유형표에 포함된 대학가 역이 없습니다. 역명을 확인하세요.")


## 추가 2. 역별 개강 효과 계산

**대학가 역별 주별 이용지수**와 **해당 역에 기존에 매칭된 비교역의 주별 이용지수**를 각각 계산합니다. 기존 `use_index`는 개강 전 일평균을 100으로 정규화해 둔 데이터입니다.

각 대학가 역에 대해 다음 값을 계산합니다.

\[\Delta_s = (\bar I_{대학,s,후}-\bar I_{대학,s,전})-(\bar I_{비교,s,후}-\bar I_{비교,s,전})\]

여기서 모든 평균은 **주별 이용지수의 평균**입니다. 이때 건대입구·합정 등 역마다 **개별 매칭 비교역을 유지**하므로 기존 연구 설계와 연결됩니다.


In [ ]:
# ===============================================================
# 추가 2. 유형별 선택 역의 주별 지수 및 역별 상대적 개강 효과
# ===============================================================
weekly_pair_frames = []
for rec in classified_pairs.itertuples(index=False):
    university_station, control_station, station_type = rec.대학역, rec.비교역, rec.역유형
    # use_index: 날짜 인덱스, 열=역명, 값=개강 전 평균 100의 이용지수
    tmp = pd.DataFrame({
        "날짜": use_index.index,
        "대학가이용지수": use_index[university_station].to_numpy(),
        "비교역이용지수": use_index[control_station].to_numpy()
    })
    tmp["주시작일"] = tmp["날짜"] - pd.to_timedelta(tmp["날짜"].dt.dayofweek, unit="D")
    w = (tmp.groupby("주시작일", as_index=False)
         .agg(대학가이용지수=("대학가이용지수", "mean"),
              비교역이용지수=("비교역이용지수", "mean"),
              관측일수=("날짜", "nunique")))
    w = w[w["관측일수"] >= MIN_DAYS_PER_WEEK].copy()
    w["개강후"] = w["주시작일"].ge(OPEN_DATE).astype(int)
    w["대학역"] = university_station
    w["비교역"] = control_station
    w["역유형"] = station_type
    w["주별차이"] = w["대학가이용지수"] - w["비교역이용지수"]
    weekly_pair_frames.append(w)

pair_weekly_types = pd.concat(weekly_pair_frames, ignore_index=True)

# 역별로 개강 전/후 주별 평균을 먼저 만든 뒤 상대적 변화량 계산
station_prepost = (
    pair_weekly_types.groupby(["역유형", "대학역", "비교역", "개강후"], as_index=False)
    .agg(대학가주별평균=("대학가이용지수", "mean"),
         비교역주별평균=("비교역이용지수", "mean"))
)
station_wide = station_prepost.pivot(
    index=["역유형", "대학역", "비교역"], columns="개강후",
    values=["대학가주별평균", "비교역주별평균"]
)
# groupby 에서 0(사전) / 1(사후) 관측 모두 있어야 함
assert station_wide.notna().all().all(), "어떤 역은 개강 전 또는 개강 후 자료가 없습니다."
station_effects = station_wide.copy()
station_effects.columns = [f"{var}_{'전' if post == 0 else '후'}" for var, post in station_effects.columns]
station_effects = station_effects.reset_index()
station_effects["대학가변화"] = station_effects["대학가주별평균_후"] - station_effects["대학가주별평균_전"]
station_effects["비교역변화"] = station_effects["비교역주별평균_후"] - station_effects["비교역주별평균_전"]
station_effects["상대적개강효과"] = station_effects["대학가변화"] - station_effects["비교역변화"]

print("[대학가 역별 개강 전후 평균 및 비교역 대비 증가 폭]")
display(station_effects.round(3).sort_values(["역유형", "대학역"]))

# 각 유형의 평균 효과는 '역별 상대적 개강 효과'를 같은 가중치로 평균
summary_types = (
    station_effects.groupby("역유형", as_index=False)
    .agg(역수=("대학역", "nunique"),
         대학가평균변화=("대학가변화", "mean"),
         비교역평균변화=("비교역변화", "mean"),
         평균상대효과=("상대적개강효과", "mean"),
         최소효과=("상대적개강효과", "min"),
         최대효과=("상대적개강효과", "max"))
)
print("[유형별 요약: 동일 유형의 대학가 역을 동일한 가중치로 평균]")
display(summary_types.round(3).sort_values("역유형"))
station_effects.to_csv(OUTPUT_DIR / "14_대학가역_유형별_개강효과_역별.csv", index=False, encoding="utf-8-sig")
summary_types.to_csv(OUTPUT_DIR / "15_대학가역_유형별_개강효과_요약.csv", index=False, encoding="utf-8-sig")


## 추가 3. 유형별 비교 그래프

그래프 A: **각 역(점)** 과 유형 평균(검은 가로선)을 함께 표시합니다. 표본이 유형당 2개뿐이므로 평균만 보면 두 역의 차이가 가려질 수 있습니다.

그래프 B: **각 유형별 대학가·매칭 비교역 주별 이용지수**를 개강 전후로 나누어 보여줍니다. 원래 연구의 주별 분석 방법을 유지합니다.


In [ ]:
# ===============================================================
# 추가 3-A. 유형별 상대 증가 폭 (각 역 점 + 평균선)
# ===============================================================
order = [x for x in ["상업", "관광", "주거"] if x in summary_types["역유형"].tolist()]
fig, ax = plt.subplots(figsize=(9, 5))
color_by_type = {"상업": "#E98F46", "관광": "#2E9E83", "주거": "#557ECB"}
for pos, category in enumerate(order):
    sub = station_effects[station_effects["역유형"] == category].sort_values("대학역")
    xx = pos + np.linspace(-.12, .12, len(sub))
    ax.scatter(xx, sub["상대적개강효과"], s=85, color=color_by_type[category], zorder=3)
    for xval, (_, rec) in zip(xx, sub.iterrows()):
        ax.annotate(rec["대학역"], (xval, rec["상대적개강효과"]), xytext=(0, 9),
                    textcoords="offset points", ha="center", fontsize=9)
    avg = sub["상대적개강효과"].mean()
    ax.hlines(avg, pos-.3, pos+.3, color="black", lw=3, label="유형 평균" if pos == 0 else None)
    ax.text(pos+.34, avg, f"{avg:+.2f}", va="center", fontsize=10)
ax.axhline(0, color="gray", ls="--", lw=1)
ax.set_xticks(range(len(order)), [f"{c} (n={len(station_effects[station_effects['역유형']==c])})" for c in order])
ax.set_ylabel("비교역 대비 개강 후 상대적 증가 (이용지수점)")
ax.set_title("대학가 역 유형별 상대적 개강 효과: 역별 값과 유형 평균")
ax.grid(axis="y", alpha=.2)
ax.legend(loc="best")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "16_유형별_역별_상대개강효과.png", dpi=170)
plt.show()

# ===============================================================
# 추가 3-B. 유형별 대학가 vs 매칭 비교역 주별 그래프
# ===============================================================
fig, axs = plt.subplots(len(order), 1, figsize=(12, 3.5*len(order)), sharex=True, squeeze=False)
for ax, category in zip(axs.flatten(), order):
    sub = pair_weekly_types[pair_weekly_types["역유형"] == category]
    weekly_type = (sub.groupby("주시작일", as_index=False)
                   .agg(대학가=("대학가이용지수", "mean"), 비교역=("비교역이용지수", "mean")))
    ax.plot(weekly_type["주시작일"], weekly_type["대학가"], marker="o", ms=3, label="대학가 역")
    ax.plot(weekly_type["주시작일"], weekly_type["비교역"], marker="o", ms=3, label="매칭 비교역")
    ax.axvline(OPEN_DATE, color="black", linestyle="--", linewidth=1, label="개강 3/4")
    ax.axhline(100, color="gray", linestyle=":", linewidth=1)
    t = summary_types.loc[summary_types["역유형"] == category].iloc[0]
    ax.set_title(f"{category} 유형: {int(t['역수'])}개 대학가 역 / 상대 효과 {t['평균상대효과']:+.2f}점")
    ax.set_ylabel("주별 이용지수")
    ax.legend(ncol=3, fontsize=9)
    ax.grid(alpha=.2)
plt.xlabel("주시작일")
plt.xticks(rotation=35)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "17_유형별_주별_대학가역_비교역_추이.png", dpi=170)
plt.show()


## 추가 4. 결과 해석과 검정의 한계

이 36개 역 분류표와 최종 매칭된 33개 대학가 역의 교집합은 **업무 0개, 상업 2개, 관광 2개, 주거 2개**, 총 6개입니다.

**따라서 유형별 '차이가 없다'라는 귀무가설에 대해 신뢰할 만한 유의확률을 제시하기 어렵습니다.** 유형별 독립 대학가 역이 2개뿐이고, 주거 유형에서는 두 역이 수락산을 공통 비교역으로 사용합니다. 주간 관측치를 수십 개로 늘려 통계검정하더라도 서로 다른 대학가 역 수가 늘어나는 것은 아니므로 표본 수를 부풀리는 *의사반복(pseudoreplication)*이 됩니다.

이 단계에서 할 수 있는 올바른 해석은 **유형별 관측된 평균 상대 효과와 역별 분포가 서로 어떠했는지**를 보여주는 탐색적 비교입니다. 이것만으로 상업·관광·주거라는 유형 자체가 개강 효과의 차이를 유발한다고 주장하지 않습니다.

추가적인 통계검정을 하려면 나머지 27개 대학가 역도 **미리 정한 재현 가능한 기준**(입지·토지이용·사업체 정보 등)으로 분류해 유형별로 충분한 수의 고유 역을 확보하고, 비교역 재사용을 반영한 검정을 설계하는 편이 좋습니다.

**보고서 예시:** “사전에 정의한 36개 역 유형표와 개강 효과 분석에 매칭된 33개 대학가 역을 교차 확인하였다. 겹치는 대학가 역 6개를 상업·관광·주거 유형으로 나누어 대학가 역과 기존 매칭 비교역의 개강 전후 주별 이용지수 변화량 차이를 계산하였다. 각 유형별 표본이 2개이므로 결과는 탐색적 기술통계로 제시하고 통계적 유의성 및 인과관계에 관한 결론은 유보하였다.”
